# Demo 2a: Brittle Patient Data Cleaning (BEFORE defensive programming)

This notebook loads patient intake data and calculates BMI. It works fine with clean data but **fails silently or cryptically** when data is messy.

**Your task:** Run this notebook with different data files and observe failures, then add defensive programming guardrails.

---

## Load and inspect data

In [1]:
import pandas as pd

# Hardcoded path - breaks if file moves or doesn't exist
df = pd.read_csv("data/patient_intake.csv")

# No validation - assumes columns exist
df.head()

,patient_id,first_name,last_name,weight_kg,height_cm,age,sex
0,P001,Mark,Johnson,91.5,177,46,M
1,P002,Donald,Walker,80.5,164,29,M
2,P003,Nancy,Rhodes,74.3,163,47,F
3,P004,Steven,Miller,64.4,171,71,M
4,P005,Javier,Johnson,72.8,178,18,M


---

## Calculate BMI

In [2]:
# No type checking - fails if values are strings
# No bounds checking - allows impossible values
df["height_m"] = df["height_cm"] / 100
df["bmi"] = df["weight_kg"] / (df["height_m"] ** 2)
df["bmi"] = df["bmi"].round(1)

df[["patient_id", "weight_kg", "height_cm", "bmi"]].head()

,patient_id,weight_kg,height_cm,bmi
0,P001,91.5,177,29.2
1,P002,80.5,164,29.9
2,P003,74.3,163,28.0
3,P004,64.4,171,22.0
4,P005,72.8,178,23.0


---

## Categorize BMI

In [3]:
# No handling for missing values
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, float("inf")],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    right=False
)

df[["patient_id", "bmi", "bmi_category"]].head(10)

,patient_id,bmi,bmi_category
0,P001,29.2,Overweight
1,P002,29.9,Overweight
2,P003,28.0,Overweight
3,P004,22.0,Normal
4,P005,23.0,Normal
5,P006,27.5,Overweight
6,P007,30.6,Obese
7,P008,29.7,Overweight
8,P009,29.1,Overweight
9,P010,20.7,Normal


---

## Filter and summarize

In [5]:
# Silent failure if filtering removes all rows
high_bmi = df[df["bmi"] > 30]
print(f"Patients with BMI > 30: {len(high_bmi)}")

# No logging - can't trace what happened during analysis
summary = df.groupby("bmi_category", observed=False)["patient_id"].count()
# Note: observed=False tells pandas to include all categories in the groupby result, even if they don't appear in the data.
print("\nBMI category distribution:")
print(summary)

Patients with BMI > 30: 14

BMI category distribution:
bmi_category
Underweight     0
Normal         15
Overweight     21
Obese          14
Name: patient_id, dtype: int64


---

## What breaks with this notebook?

**Try running with these data files and observe the failures:**

1. **`patient_intake_missing_height.csv`** - Missing `height_cm` column
   - Error: `KeyError: 'height_cm'`
   - Problem: No schema validation before processing

2. **`patient_intake_bad_values.csv`** - Out-of-bounds values (500 kg weight, 50 cm height)
   - Silent failure: BMI calculations produce nonsense
   - Problem: No bounds checking

3. **Non-existent file:** Change path to `data/wrong_file.csv`
   - Error: `FileNotFoundError` with cryptic message
   - Problem: No existence check, no helpful error

**Other issues:**
- No logging means you can't trace execution or debug production issues
- If you add print/logging later, be careful not to expose PHI (patient IDs, names, etc.)

---

## How to do better? (Demo 2b)

Add defensive programming to this notebook:

1. **Config instead of hardcoded paths:** Load file path and bounds from `02_config.yaml`
2. **Schema validation:** Check required columns exist before processing
3. **Bounds checking:** Validate weight, height, age are in realistic ranges
4. **Logging:** Add `logging.info()` to trace execution
5. **Graceful error handling:** Raise informative exceptions with context

Run the hardened version with all three data files—it should fail fast with clear error messages instead of silently producing wrong results.